# 03 · Batching

`ServiceBusMessageBatch` lets you send many messages in **one network round-trip**, while respecting the namespace's per-batch size limit (256 KB on Standard, 1 MB on Premium).

The batch object is created by the **sender** so it can size itself correctly. You call `TryAddMessage` until it returns `false`, then send.


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.QueueName);

## 1. Build a batch, flushing whenever it fills up

In [ ]:
var toSend = Enumerable.Range(1, 5_000)
    .Select(i => new ServiceBusMessage($"message-{i:D5}"))
    .ToList();

int totalSent = 0;
int batchNumber = 0;
int i2 = 0;

while (i2 < toSend.Count)
{
    using ServiceBusMessageBatch batch = await sender.CreateMessageBatchAsync();

    if (!batch.TryAddMessage(toSend[i2]))
        throw new Exception($"Message {i2} is too large for an empty batch.");
    i2++;

    while (i2 < toSend.Count && batch.TryAddMessage(toSend[i2])) i2++;

    await sender.SendMessagesAsync(batch);
    totalSent += batch.Count;
    batchNumber++;
    Console.WriteLine($"Batch #{batchNumber}: sent {batch.Count} (running total {totalSent})");
}

Console.WriteLine($"\nSent {totalSent} messages in {batchNumber} batches.");

## 2. Why not just call `SendMessagesAsync(IEnumerable<...>)`?

You can — the SDK will batch for you. The explicit batch API is useful when:

- You want to **know up front** whether a single message would exceed the limit.
- You're streaming messages and want to flush opportunistically.
- You need fine control to mix entity-targeted batches.

## 3. Drain (so the queue is clean for later notebooks)

In [ ]:
var receiver = client.CreateReceiver(Config.QueueName, new ServiceBusReceiverOptions
{
    ReceiveMode = ServiceBusReceiveMode.ReceiveAndDelete
});

int drained = 0;
while (true)
{
    var batch = await receiver.ReceiveMessagesAsync(maxMessages: 500, maxWaitTime: TimeSpan.FromSeconds(2));
    if (batch.Count == 0) break;
    drained += batch.Count;
}
Console.WriteLine($"Drained {drained} messages.");

await receiver.DisposeAsync();
await sender.DisposeAsync();
await client.DisposeAsync();

Next: [`04-processor.ipynb`](04-processor.ipynb)